# 05 - Score Answer Quality Carefully

Evaluates generated answer quality. The default path is offline and uses an extractive context answer plus lexical relevance. Optional DeepEval scoring is available when API access is desired.


## Learning Goal

Move from evidence retrieval to answer quality without pretending that a vague quality score is enough. This lab starts with lightweight offline proxies and keeps LLM-as-judge scoring optional so students can see the cost and calibration tradeoff.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This is the point where evaluation becomes more subjective. The main lesson is to be cautious: answer-quality evals need clear criteria, human labels, or careful judge validation.

## Related AI Evals Concepts

- How To Trust A LLM Judge: judge scores need calibration against human labels before they become decision metrics.
- Don't Use Likert Scales: prefer targeted pass/fail or specific sub-metrics over broad 1-5 quality ratings.
- Types Of Automated Evals: combine cheap proxy checks with optional LLM-as-judge scoring.
- AI Eval Mistakes: do not outsource judgment to a model before understanding the data and failure modes.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import importlib
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing, start_span  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [ ]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_FILE = TRACE_DIR / "05_answer_quality_deepeval.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


## Load Answer-Quality Test Sets


In [ ]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


In [ ]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


## Define Answer-Quality Proxies And Optional Judge


In [ ]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

RUN_DEEPEVAL = True

async def answer_one(row: pd.Series, top_k: int = 3) -> dict[str, Any]:
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    expected_answer = str(row.get("expected_answer") or "")

    with start_span("notebook.answer_quality.row", dataset=str(row["dataset"]), query_length=len(query)):
        context_docs: list[str] = []
        if is_local_source(expected_source):
            retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
            docs = await retriever.retrieve(SourceType(expected_source), query)
            context_docs = [doc.text for doc in docs]

        actual_answer = preview_text(context_docs[0], max_words=50) if context_docs else ""
        lexical_question_relevance = lexical_score(query, actual_answer) if actual_answer else None
        lexical_expected_overlap = lexical_score(expected_answer, actual_answer) if expected_answer and actual_answer else None

        deepeval_score = None
        deepeval_reason = ""
        deepeval_success = None
        if RUN_DEEPEVAL and actual_answer:
            from deepeval.metrics import AnswerRelevancyMetric
            from deepeval.test_case import LLMTestCase

            metric = AnswerRelevancyMetric(threshold=0.7, model=settings.evaluator_model, include_reason=True, async_mode=False)
            test_case = LLMTestCase(
                input=query,
                actual_output=actual_answer,
                expected_output=expected_answer or None,
                retrieval_context=context_docs or None,
            )
            metric.measure(test_case)
            deepeval_score = metric.score
            deepeval_reason = metric.reason
            deepeval_success = metric.success

        return {
            "dataset": row["dataset"],
            "query": query,
            "expected_source_type": expected_source,
            "actual_answer": actual_answer,
            "expected_answer": expected_answer,
            "context_count": len(context_docs),
            "lexical_question_relevance": lexical_question_relevance,
            "lexical_expected_overlap": lexical_expected_overlap,
            "answer_relevancy": deepeval_score,
            "answer_relevancy_reason": deepeval_reason,
            "answer_relevancy_success": deepeval_success,
            "metric_status": "deepeval_scored" if deepeval_score is not None else ("offline_proxy" if actual_answer else "skipped_no_local_context"),
        }


async def evaluate_answers(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        rows.append(await answer_one(row))
    return pd.DataFrame(rows)


## Start With Offline Answer-Quality Proxies


In [ ]:
base_answer_results = await evaluate_answers(base_df)
base_answer_results.head()


In [ ]:
base_answer_summary = base_answer_results.groupby(["dataset", "expected_source_type", "metric_status"], dropna=False).agg(
    examples=("query", "count"),
    lexical_question_relevance=("lexical_question_relevance", "mean"),
    lexical_expected_overlap=("lexical_expected_overlap", "mean"),
    answer_relevancy=("answer_relevancy", "mean"),
).reset_index()
base_answer_summary


## Inspect Hard Answer-Quality Cases


In [ ]:
challenge_answer_results = await evaluate_answers(challenge_df)
challenge_answer_results.head()


In [ ]:
challenge_answer_results[["query", "expected_source_type", "actual_answer", "lexical_question_relevance", "metric_status"]]


## Pay Attention To

- Answer quality is harder to score than routing or retrieval because it can be subjective.
- Lightweight lexical proxies are useful for iteration, but they are not a substitute for human judgment.
- LLM-as-judge evaluation should be optional until you can compare it against trusted labels.
- Keep the question narrow: score a specific failure mode rather than overall helpfulness.


## Optional Advanced Path: DeepEval LLM Judge

The `RUN_DEEPEVAL` flag in the helper cell enables model-graded answer relevancy. Treat this as an advanced path: it adds cost and latency, and its scores should be checked against human labels before being used as a source of truth.


## Export Answer-Quality Artifacts


In [ ]:
answer_results = pd.concat([base_answer_results, challenge_answer_results], ignore_index=True)
answer_summary = pd.concat([base_answer_summary], ignore_index=True)

results_path = PROJECT_ROOT / "output/answer_quality_results.csv"
summary_path = PROJECT_ROOT / "output/answer_quality_summary.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.parent.mkdir(parents=True, exist_ok=True)
answer_results.to_csv(results_path, index=False)
answer_summary.to_csv(summary_path, index=False)

results_path, summary_path
